<a href="https://colab.research.google.com/github/andrewlivingstonvj/ai-data-analyst-agent/blob/main/AIAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q google-genai pandas


In [28]:
from google import genai
client = genai.Client(api_key="YOUR API KEY")

In [29]:
import pandas as pd
df = pd.read_csv("/content/SuperStoreOrders - SuperStoreOrders.csv")   # <-- change this to your actual filename
print(df.shape)
df.head()

(51290, 21)


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.0,106.140,35.46,Medium,2011
1,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.1,36.036,9.72,Medium,2011
2,HU-2011-1220,1/1/2011,5/1/2011,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.0,29.640,8.17,High,2011
3,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,Office Supplies,Paper,"Enermax Note Cards, Premium",45,3,0.5,-26.055,4.82,High,2011
4,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114,5,0.1,37.770,4.70,Medium,2011


In [21]:
def ask_data(question):
    schema_info = f"Columns: {list(df.columns)}\nSample rows:\n{df.head(3).to_string()}"

    prompt = f"""
You are a data analyst. Here is a pandas dataframe called df.
{schema_info}

Question: {question}

Write ONLY python pandas code (no explanation, no markdown).
The LAST line of your code MUST be: result = <your computed value>
Example: result = df['sales'].sum()

Do not just write an expression on its own — always assign it to result.
"""

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    code = response.text.replace("```python", "").replace("```", "").strip()

    local_vars = {"df": df, "pd": pd}
    try:
        exec(code, {}, local_vars)
        if "result" not in local_vars:
            # AI forgot to assign result — grab the last line ourselves as a backup
            last_line = code.strip().split("\n")[-1]
            if "=" not in last_line:
                local_vars["result"] = eval(last_line, {}, local_vars)
        return local_vars.get("result"), code
    except Exception as e:
        return f"Error running generated code: {e}", code

In [30]:
answer, code_used = ask_data("Which region had the highest total sales?")
print("Answer:", answer)
print("\nCode the AI wrote:\n", code_used)

Answer: EMEA

Code the AI wrote:
 df.groupby('region')['sales'].sum().idxmax()
result = df.groupby('region')['sales'].sum().idxmax()


In [31]:
answer2, code3 = ask_data("Which year had the lowest sales?")
print("Answer:", answer3)
print("\nCode used:\n", code3)

Answer: 2013

Code used:
 result = df.groupby('year')['sales'].sum().idxmin()
